# LoRA GPT-2 Medium E2E Evaluation on Google Colab

Use this notebook after training has finished and the run folder has been copied to Google Drive.

It will:

- clone this project,
- install dependencies,
- download and preprocess the E2E test data,
- load the trained LoRA adapter,
- generate the full E2E test predictions on GPU,
- compute quick BLEU and ROUGE-L metrics,
- create report figures,
- copy evaluation outputs back to Drive.

Recommended runtime: `Runtime > Change runtime type > GPU`. A faster GPU such as A100 or L4 is strongly preferred for full beam-search generation.

## 1. Check GPU

In [1]:
!nvidia-smi

import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

Sat Apr 25 21:06:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone Or Update The Repo

If the repository is private, paste a GitHub token when prompted. If it is public, press Enter.

In [2]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "main"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("working directory:", Path.cwd())

GitHub token, or press Enter for public clone: ··········
working directory: /content/CS4782-final-project/lora-gpt2-medium-e2e


## 3. Install Dependencies

In [3]:
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.7 MB/s eta 0:00:00


## 4. Mount Drive And Locate The Trained Adapter

The training notebook used `/content/drive/MyDrive/e2e_lora_r4_alpha32` as the default backup path. If your run folder is elsewhere, update `DRIVE_RUN_HINT` before running the cell.

In [4]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

DRIVE_RUN_HINT = Path("/content/drive/MyDrive/e2e_lora_r4_alpha32")
LOCAL_RUN_DIR = Path("outputs/runs/e2e_lora_r4_alpha32")

candidates = []
for root in [DRIVE_RUN_HINT, Path("/content/drive/MyDrive")]:
    if root.exists():
        candidates.extend(root.rglob("adapter_final.pt"))

if not candidates:
    raise FileNotFoundError(
        "Could not find adapter_final.pt in Drive. Update DRIVE_RUN_HINT to your backed-up run folder."
    )

adapter_path = sorted(candidates, key=lambda path: len(path.parts))[0]
drive_run_dir = adapter_path.parents[1]
LOCAL_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(drive_run_dir, LOCAL_RUN_DIR, dirs_exist_ok=True)

LOCAL_ADAPTER = LOCAL_RUN_DIR / "checkpoints" / "adapter_final.pt"
print("Drive run dir:", drive_run_dir)
print("Local adapter:", LOCAL_ADAPTER)
print("Adapter exists:", LOCAL_ADAPTER.exists())

Mounted at /content/drive
Drive run dir: /content/drive/MyDrive/e2e_lora_r4_alpha32
Local adapter: outputs/runs/e2e_lora_r4_alpha32/checkpoints/adapter_final.pt
Adapter exists: True


## 5. Download And Preprocess E2E Data

In [5]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml
!wc -l data/raw/e2e/*.txt data/processed/e2e_gpt2/*.jsonl

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9398k  100 9398k    0     0  7674k      0  0:00:01  0:00:01 --:--:-- 7672k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1170k  100 1170k    0     0  1592k      0 --:--:-- --:--:-- --:--:-- 1594k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1319k  100 1319k    0     0  2220k      0 --:--:-- --:--:-- --:--:-- 2221k
config.json: 100% 718/718 [00:00<00:00, 3.03MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 134kB/s]
vocab.json: 1.04MB [00:00, 1.93MB/s]
merges.txt: 456kB [00:00, 480kB/s]
tokenizer.json: 1.36MB [00:00, 31.4MB/s]
wrote 42061 examples to /content/CS4782-final-project/lora-gpt2-m

## 6. Generate Full Test Predictions

This is the slow step. `BATCH_SIZE=4` is conservative for beam size 10. If Colab gives you a large GPU, try `8` or `16`. If you hit out-of-memory, lower it.

In [6]:
BATCH_SIZE = 4

!TOKENIZERS_PARALLELISM=false python scripts/generate.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --split test \
  --adapter "$LOCAL_ADAPTER" \
  --batch-size "$BATCH_SIZE"

!ls -lh outputs/runs/e2e_lora_r4_alpha32/generations_test.txt
!python - <<'PY'
from pathlib import Path
path = Path('outputs/runs/e2e_lora_r4_alpha32/generations_test.txt')
print('prediction lines:', sum(1 for _ in path.open()))

model.safetensors: 100% 1.52G/1.52G [00:04<00:00, 360MB/s]
Loading weights: 100% 292/292 [00:00<00:00, 999.46it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generation_config.json: 100% 124/124 [00:00<00:00, 554kB/s]
generating test:   0% 0/1174 [00:00<?, ?batch/s]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
generating test:   0% 1/1174 [00:03<59:42,  3.05s/batch]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
generating test:   0% 2/1174 [00:05<50:21,  

NameError: name 'PY' is not defined

## 7. Run Metrics

In [7]:
!python scripts/evaluate.py --config configs/e2e_gpt2_medium_lora.yaml
!cat outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json

wrote 4693 references to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/references_test.txt
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
wrote metrics to /content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json
{
  "bleu": 14.410638503847743,
  "num_examples": 4693,
  "rouge_l": 37.04713176923605
}

## 9. Create Figures

In [8]:
!python scripts/make_figures.py --config configs/e2e_gpt2_medium_lora.yaml
!ls -lh figures
!cat figures/summary.json

wrote figures to /content/CS4782-final-project/lora-gpt2-medium-e2e/figures
wrote summary to /content/CS4782-final-project/lora-gpt2-medium-e2e/figures/summary.json
total 208K
-rw-r--r-- 1 root root 51K Apr 25 21:53 epoch_loss.png
-rw-r--r-- 1 root root 25K Apr 25 21:53 evaluation_metrics.png
-rw-r--r-- 1 root root 830 Apr 25 21:53 summary.json
-rw-r--r-- 1 root root 37K Apr 25 21:53 trainable_parameters.png
-rw-r--r-- 1 root root 84K Apr 25 21:53 training_loss.png
{
  "epoch_loss": {
    "epoch_1_mean_loss": 2.8623015162789716,
    "epoch_2_mean_loss": 2.646173942220578,
    "epoch_3_mean_loss": 2.605156097724076,
    "epoch_4_mean_loss": 2.582705746770767,
    "epoch_5_mean_loss": 2.570376970143135
  },
  "figures_dir": "/content/CS4782-final-project/lora-gpt2-medium-e2e/figures",
  "metrics": {
    "bleu": 14.410638503847743,
    "rouge_l": 37.04713176923605
  },
  "parameters": {
    "full_finetune_trainable_params": 355216384.0,
    "lora_trainable_params": 393216.0,
    "trainabl

## 10. Back Up Evaluation Outputs To Drive

In [9]:
import shutil
from pathlib import Path

drive_eval_dir = drive_run_dir / "evaluation_outputs"
drive_eval_dir.mkdir(parents=True, exist_ok=True)

for path in [
    Path("outputs/runs/e2e_lora_r4_alpha32/generations_test.txt"),
    Path("outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json"),
    Path("data/processed/e2e_gpt2/references_test.txt"),
]:
    if path.exists():
        shutil.copy2(path, drive_eval_dir / path.name)

if Path("figures").exists():
    shutil.copytree("figures", drive_eval_dir / "figures", dirs_exist_ok=True)

print("Backed up evaluation outputs to:", drive_eval_dir)
!find "$drive_eval_dir" -maxdepth 2 -type f -print

Backed up evaluation outputs to: /content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/generations_test.txt
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/generations_test.metrics.json
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/references_test.txt
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/figures/trainable_parameters.png
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/figures/evaluation_metrics.png
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/figures/training_loss.png
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/figures/summary.json
/content/drive/MyDrive/e2e_lora_r4_alpha32/evaluation_outputs/figures/epoch_loss.png
